# Lab #2 — Robot Motion Planning and Control

**Selected manipulator:** Stanford arm from Robotics Toolbox  
**Goal:** solve the forward and inverse kinematics problems, construct the workspace, and compare three trajectory planning methods.

This notebook follows the assignment step by step and contains explanatory text before each code cell.

## 1. Import the required libraries

In this step we import the numerical, plotting, and robotics libraries that will be used in the report.

In [ ]:
from math import pi
import numpy as np
import pandas as pd
import roboticstoolbox as rtb
import matplotlib.pyplot as plt
from spatialmath import SE3

plt.rcParams["figure.figsize"] = (8, 5)
np.set_printoptions(precision=4, suppress=True)

## 2. Load the manipulator model and fill all robot parameters

The assignment requires a manipulator different from Puma560.  
Here the **Stanford manipulator** is used.

The built-in Stanford model already contains geometric and dynamic data.  
For drive parameters that may be absent or zero, simple engineering values are assigned so that the model is fully specified:
- motor inertia `Jm`
- viscous friction `B`
- Coulomb friction `Tc`
- gear ratio `G`
- joint limits `qlim`

In [ ]:
robot = rtb.models.DH.Stanford()

# Default values used only when a parameter is missing or equal to zero.
default_Jm = [0.01, 0.01, 0.01, 0.005, 0.005, 0.005]
default_B  = [0.001, 0.001, 0.001, 0.0005, 0.0005, 0.0005]
default_Tc = [
    np.array([0.05, -0.05]),
    np.array([0.05, -0.05]),
    np.array([1.00, -1.00]),   # prismatic joint: force units
    np.array([0.02, -0.02]),
    np.array([0.02, -0.02]),
    np.array([0.02, -0.02]),
]
default_G = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
default_qlim = [
    np.array([-pi, pi]),
    np.array([-pi/2, pi/2]),
    np.array([0.0, 0.5]),
    np.array([-pi, pi]),
    np.array([-pi/2, pi/2]),
    np.array([-pi, pi]),
]

for i, link in enumerate(robot.links):
    if getattr(link, "Jm", None) in [None, 0]:
        link.Jm = default_Jm[i]
    if getattr(link, "B", None) in [None, 0]:
        link.B = default_B[i]

    Tc_value = getattr(link, "Tc", None)
    if Tc_value is None or np.allclose(np.asarray(Tc_value, dtype=float), 0.0):
        link.Tc = default_Tc[i]

    if getattr(link, "G", None) in [None, 0]:
        link.G = default_G[i]

    qlim_value = getattr(link, "qlim", None)
    if qlim_value is None or np.any(np.isnan(np.asarray(qlim_value, dtype=float))):
        link.qlim = default_qlim[i]

print(robot)

## 3. Output the Denavit–Hartenberg parameters

This step prints the DH table of the selected manipulator.

In [ ]:
dh_rows = []
for i, link in enumerate(robot.links, start=1):
    joint_type = "R" if link.sigma == 0 else "P"
    dh_rows.append({
        "Joint": i,
        "Type": joint_type,
        "a": float(link.a),
        "alpha": float(link.alpha),
        "d": float(link.d),
        "theta": float(link.theta),
        "offset": float(link.offset),
        "qlim_min": float(link.qlim[0]),
        "qlim_max": float(link.qlim[1]),
    })

dh_table = pd.DataFrame(dh_rows)
dh_table

## 4. Output the full dynamic parameters of the robot

The table below contains:
- link masses `m`
- centers of mass `r`
- inertia tensors `I`
- drive inertias `Jm`
- viscous friction `B`
- Coulomb friction `Tc`
- gear ratios `G`
- generalized-coordinate constraints `qlim`

In [ ]:
dyn_rows = []
for i, link in enumerate(robot.links, start=1):
    dyn_rows.append({
        "Joint": i,
        "Type": "R" if link.sigma == 0 else "P",
        "m": float(link.m),
        "r": np.array(link.r, dtype=float),
        "I": np.array(link.I, dtype=float),
        "Jm": float(link.Jm),
        "B": float(link.B),
        "Tc": np.array(link.Tc, dtype=float),
        "G": float(link.G),
        "qlim": np.array(link.qlim, dtype=float),
    })

dyn_table = pd.DataFrame(dyn_rows)
dyn_table

## 5. Set an arbitrary initial robot configuration

An arbitrary initial configuration is chosen inside the joint limits.

In [ ]:
q_start = np.array([0.20, -0.60, 0.20, 0.30, -0.20, 0.40])
print("Initial configuration q_start:")
print(q_start)

## 6. Display the initial robot configuration

This cell plots the manipulator in the initial configuration.

In [ ]:
robot.plot(q_start, limits=[-1, 1, -1, 1, -0.1, 1.2])
plt.show()

## 7. Solve the forward kinematics problem

For the chosen generalized coordinates `q_start`, the end-effector pose is found by forward kinematics.

In [ ]:
T_start = robot.fkine(q_start)
print("Forward kinematics result for q_start:")
print(T_start)

## 8. Construct the manipulator workspace under joint constraints

To visualize the reachable workspace, the first three joints are sampled within their limits.  
For the Stanford arm, these joints mainly define the end-effector position, while the wrist joints primarily affect orientation.

In [ ]:
n_samples = 15

q1_vals = np.linspace(robot.links[0].qlim[0], robot.links[0].qlim[1], n_samples)
q2_vals = np.linspace(robot.links[1].qlim[0], robot.links[1].qlim[1], n_samples)
q3_vals = np.linspace(robot.links[2].qlim[0], robot.links[2].qlim[1], n_samples)

workspace_points = []

for q1 in q1_vals:
    for q2 in q2_vals:
        for q3 in q3_vals:
            q_tmp = np.array([q1, q2, q3, 0, 0, 0])
            T_tmp = robot.fkine(q_tmp)
            workspace_points.append(T_tmp.t)

workspace_points = np.array(workspace_points)
workspace_points.shape

## 9. Plot the workspace

The cloud of red points represents the sampled workspace of the manipulator.

In [ ]:
fig = plt.figure(figsize=(8, 8), dpi=150)
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    workspace_points[:, 0],
    workspace_points[:, 1],
    workspace_points[:, 2],
    s=2,
)
ax.set_title("Sampled workspace of the Stanford manipulator")
ax.set_xlabel("x, m")
ax.set_ylabel("y, m")
ax.set_zlabel("z, m")
plt.show()

## 10. Select an end point inside the workspace

A reachable final end-effector pose is selected by first choosing a valid reference joint configuration and then computing its forward kinematics.  
This guarantees that the target lies inside the workspace.

In [ ]:
q_goal_reference = np.array([1.00, 0.35, 0.38, -0.50, 0.45, -0.30])
T_target = robot.fkine(q_goal_reference)

print("Chosen target end-effector pose:")
print(T_target)
print("\nChosen target position [x, y, z] in meters:")
print(T_target.t)

## 11. Solve the inverse kinematics problem for the selected end point

The inverse kinematics solution is calculated for the target pose.  
Because the target was generated from a valid configuration, a feasible solution is expected.

In [ ]:
ik_result = robot.ikine_LM(T_target, q0=q_goal_reference)
q_end = ik_result.q

print("IK success:", ik_result.success)
print("IK residual:", ik_result.residual)
print("Final joint configuration q_end:")
print(q_end)

## 12. Verify the inverse-kinematics result and display the final configuration

The pose obtained from `q_end` is compared with the target pose, and the final robot configuration is shown.

In [ ]:
T_end_check = robot.fkine(q_end)

print("Pose from q_end:")
print(T_end_check)

print("\nPosition error norm:")
print(np.linalg.norm(T_target.t - T_end_check.t))

robot.plot(q_end, limits=[-1, 1, -1, 1, -0.1, 1.2])
plt.show()

## 13. Plan the trajectory between the initial and final positions

Three planning methods are used:
1. `jtraj`
2. `mtraj` with trapezoidal velocity profile
3. `mtraj` with quintic polynomial profile

The trajectory is planned in joint space between `q_start` and `q_end`.

In [ ]:
N = 100
t0 = 0.0
tf = 5.0
time = np.linspace(t0, tf, N)

tr_jtraj = rtb.jtraj(q_start, q_end, time)
tr_trap  = rtb.mtraj(rtb.trapezoidal, q_start, q_end, time)
tr_quint = rtb.mtraj(rtb.quintic, q_start, q_end, time)

print("Trajectory generation completed.")

## 14. Plot the position graphs of all robot joints

The first set of graphs compares joint positions for the three planning methods.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=150)
axes = axes.ravel()

for j in range(robot.n):
    axes[j].plot(time, tr_jtraj.q[:, j], label="jtraj", linewidth=2)
    axes[j].plot(time, tr_trap.q[:, j],  label="trapezoidal", linewidth=2)
    axes[j].plot(time, tr_quint.q[:, j], label="quintic", linewidth=2)
    axes[j].set_title(f"Joint {j+1} position")
    axes[j].set_xlabel("Time, s")
    axes[j].set_ylabel("q (rad or m)")
    axes[j].grid(True)
    axes[j].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 15. Plot the velocity graphs of all robot joints

The second set of graphs compares joint velocities for the three planning methods.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=150)
axes = axes.ravel()

for j in range(robot.n):
    axes[j].plot(time, tr_jtraj.qd[:, j], label="jtraj", linewidth=2)
    axes[j].plot(time, tr_trap.qd[:, j],  label="trapezoidal", linewidth=2)
    axes[j].plot(time, tr_quint.qd[:, j], label="quintic", linewidth=2)
    axes[j].set_title(f"Joint {j+1} velocity")
    axes[j].set_xlabel("Time, s")
    axes[j].set_ylabel("q_dot (rad/s or m/s)")
    axes[j].grid(True)
    axes[j].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 16. Plot the acceleration graphs of all robot joints

The third set of graphs compares joint accelerations for the three planning methods.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=150)
axes = axes.ravel()

for j in range(robot.n):
    axes[j].plot(time, tr_jtraj.qdd[:, j], label="jtraj", linewidth=2)
    axes[j].plot(time, tr_trap.qdd[:, j],  label="trapezoidal", linewidth=2)
    axes[j].plot(time, tr_quint.qdd[:, j], label="quintic", linewidth=2)
    axes[j].set_title(f"Joint {j+1} acceleration")
    axes[j].set_xlabel("Time, s")
    axes[j].set_ylabel("q_ddot (rad/s^2 or m/s^2)")
    axes[j].grid(True)
    axes[j].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 17. Optional animation of the three trajectories

This step is optional.  
It allows the motion planned by each method to be visualized.

In [ ]:
# Uncomment the lines below if you want to animate the motion.
# robot.plot(tr_jtraj.q, limits=[-1, 1, -1, 1, -0.1, 1.2], dt=tf/N)
# robot.plot(tr_trap.q,  limits=[-1, 1, -1, 1, -0.1, 1.2], dt=tf/N)
# robot.plot(tr_quint.q, limits=[-1, 1, -1, 1, -0.1, 1.2], dt=tf/N)

## 18. Conclusion

In this laboratory work, the Stanford manipulator was loaded from the Robotics Toolbox and used instead of Puma560.  
The DH parameters and the dynamic parameters of the robot were presented, including masses, centers of mass, inertia tensors, drive inertias, friction coefficients, gear ratios, and joint limits.

A valid initial configuration was selected and the forward kinematics problem was solved for it.  
Then the manipulator workspace was constructed numerically by sampling the constrained joint space.  
A reachable final end-effector pose was selected inside this workspace, and the inverse kinematics problem was solved successfully.

Finally, trajectories between the initial and final configurations were generated using three methods: `jtraj`, trapezoidal `mtraj`, and quintic `mtraj`.  
The plots of position, velocity, and acceleration for all joints showed the differences between the planning methods.  
Thus, the laboratory work demonstrated the practical use of forward kinematics, inverse kinematics, workspace analysis, and trajectory planning for a serial manipulator.